# ARK-020 V4 — hardened A1.3 operator launcher

Pinned to the exact A1.3 executable used by the current Drive campaign. This launcher also verifies that the immutable ARK-018 substrate is visible in the mounted Drive before attempting a resume. If Colab has a stale Drive mount, it performs one forced remount and fails closed if the substrate is still unavailable.


In [ ]:
import json, subprocess, sys, os, shutil
from pathlib import Path

PINNED_RUNNER_COMMIT = '4e9b49dfad00c52e569efecff14b887f1dd0369e'
BRANCH = 'arkenstone-ark020-v4'
REPO = '/content/An-Ra-the-new-AGI-ark020v4'
REPO_URL = 'https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
ROOT = Path('/content/drive/MyDrive/genisis-arkenstone/ARK020_V4_CONTINUAL')
ARK018_PREPARED = Path('/content/drive/MyDrive/genisis-arkenstone/ARK018_SCIENCE_BIRTH_V1/prepared')
ARK018_REQUIRED = (
    'ARK-018_PREPARED_RECEIPT.json',
    'tokenizer.json',
    'token_counts.npy',
    'train.bin',
    'control.bin',
    'sealed.bin',
)
SCAN_GATE_PASS = False
SAFE_ACTION = None

print('=== ARK-020 V4 A1.3 CELL 0: DRIVE / SUBSTRATE / PIN / RESUME SCAN ===')
from google.colab import drive

def required_missing():
    return [name for name in ARK018_REQUIRED if not (ARK018_PREPARED / name).is_file()]

drive.mount('/content/drive', force_remount=False)
if not Path('/content/drive/MyDrive').is_dir():
    raise SystemExit('SAFE ACTION: STOP — DRIVE UNAVAILABLE')

missing = required_missing()
if missing:
    print('ARK-018 substrate is not visible through the current Colab Drive mount.')
    print('Missing:', missing)
    print('Forcing one clean Drive remount...')
    try:
        drive.flush_and_unmount()
    except Exception as exc:
        print('flush_and_unmount note:', repr(exc))
    drive.mount('/content/drive', force_remount=True)
    missing = required_missing()

if missing:
    raise SystemExit(
        'SAFE ACTION: STOP — ARK-018 SUBSTRATE STILL NOT VISIBLE AFTER REMOUNT. '
        'Mount the Google Drive account that contains '
        'MyDrive/genisis-arkenstone/ARK018_SCIENCE_BIRTH_V1/prepared. '
        'Missing: ' + ', '.join(missing)
    )

print('ARK-018 SUBSTRATE VISIBILITY: PASS')
for name in ARK018_REQUIRED:
    p = ARK018_PREPARED / name
    print(' ', name, p.stat().st_size, 'bytes')
print('CAMPAIGN ROOT:', ROOT)
print('CAMPAIGN ROOT EXISTS:', ROOT.exists())

if os.path.exists(REPO) and not os.path.isdir(os.path.join(REPO, '.git')):
    shutil.rmtree(REPO)
if not os.path.exists(REPO):
    subprocess.run(['git','clone','--depth','100','--branch',BRANCH,REPO_URL,REPO], check=True)
else:
    subprocess.run(['git','-C',REPO,'remote','set-url','origin',REPO_URL], check=True)
    subprocess.run(['git','-C',REPO,'fetch','--depth','100','origin',BRANCH], check=True)
subprocess.run(['git','-C',REPO,'reset','--hard'], check=True)
subprocess.run(['git','-C',REPO,'clean','-fd'], check=True)
subprocess.run(['git','-C',REPO,'checkout','--detach',PINNED_RUNNER_COMMIT], check=True)
head = subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'], text=True).strip()
assert head == PINNED_RUNNER_COMMIT, (head, PINNED_RUNNER_COMMIT)
print('PINNED A1.3 COMMIT OK:', head)

runner = os.path.join(REPO, 'experiments/ARK-020-V4/run_ark020_v4_hardened.py')
scan = subprocess.run(
    [sys.executable, runner, '--mode','scan','--drive-ok','True'],
    cwd=REPO, capture_output=True, text=True
)
print(scan.stdout)
if scan.stderr.strip():
    print('--- scan stderr ---')
    print(scan.stderr)
if scan.returncode != 0:
    raise SystemExit(f'SAFE ACTION: STOP — SCAN COMMAND FAILED ({scan.returncode})')
marker = '@@SCAN_JSON@@'
marker_line = next((ln for ln in scan.stdout.splitlines() if ln.startswith(marker)), None)
if marker_line is None:
    raise SystemExit('SAFE ACTION: STOP — SCAN DID NOT EMIT @@SCAN_JSON@@')
scan_info = json.loads(marker_line[len(marker):])
SAFE_ACTION = scan_info.get('SAFE_ACTION')
print('PARSED SAFE ACTION:', SAFE_ACTION)
if SAFE_ACTION == 'RESUME':
    print('AUTO-RESUME AUTHORIZED')
    print('ACTIVE ARM:', scan_info.get('ACTIVE_ARM'))
    print('ACTIVE PHASE:', scan_info.get('ACTIVE_PHASE'))
    print('SAVED PHASE STEP:', scan_info.get('SAVED_PHASE_STEP'))
    print('COMPLETED ARMS:', scan_info.get('COMPLETED_ARMS'), '/', scan_info.get('TOTAL_ARMS'))
elif SAFE_ACTION == 'START NEW CAMPAIGN':
    print('No compatible prior checkpoint/results found; new campaign is authorized.')
else:
    raise SystemExit('Not safe to continue: ' + str(SAFE_ACTION))
SCAN_GATE_PASS = True
print('CELL 0 GATE: PASS')


In [ ]:
import subprocess, sys, torch, os

assert globals().get('SCAN_GATE_PASS') is True, 'Run Cell 0 successfully first.'
TEST_GATE_PASS = False
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU before running.'
print('ATTACHED GPU:', torch.cuda.get_device_name(0), '| torch', torch.__version__)

compile_paths = [
    'experiments/ARK-020-V4/ark020_v4_core.py',
    'experiments/ARK-020-V4/run_ark020_v4.py',
    'experiments/ARK-020-V4/ark020_v4_durability.py',
    'experiments/ARK-020-V4/ark020_v4_a1_guardrails.py',
    'experiments/ARK-020-V4/ark020_v4_device_guard.py',
    'experiments/ARK-020-V4/ark020_v4_template_guard.py',
    'experiments/ARK-020-V4/run_ark020_v4_hardened.py',
    'tests/conftest.py',
    'tests/test_ark020_v4_a13_template_guard.py',
]
for p in compile_paths:
    subprocess.run([sys.executable, '-m', 'py_compile', os.path.join(REPO, p)], check=True)
print('compile gate: PASS on', len(compile_paths), 'files')

test_env = dict(os.environ)
test_env['CUDA_VISIBLE_DEVICES'] = ''
test_env['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
test_env['PYTHONPATH'] = REPO + os.pathsep + test_env.get('PYTHONPATH', '')
print('Running A1/A1.1/A1.2/A1.3 durability contracts (CPU-isolated)...')
a1 = subprocess.run(
    [sys.executable, '-m', 'pytest',
     'tests/test_ark020_v4_durability.py',
     'tests/test_ark020_v4_a1_guardrails.py',
     'tests/test_ark020_v4_a13_template_guard.py', '-q'],
    cwd=REPO, env=test_env
)
if a1.returncode != 0:
    raise SystemExit('Durability contract suite failed — DO NOT RUN campaign')
print('Running inherited V4 suite (CPU-isolated)...')
v4 = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/test_ark020_v4.py', '-q'],
    cwd=REPO, env=test_env
)
if v4.returncode != 0:
    raise SystemExit('Inherited V4 suite failed — DO NOT RUN campaign')
assert torch.cuda.is_available(), 'T4 disappeared after tests.'
print('GPU STILL ATTACHED:', torch.cuda.get_device_name(0))
TEST_GATE_PASS = True
print('CELL 1 GATE: PASS')


In [ ]:
# Full campaign. Existing compatible Drive state is resumed automatically.
import os, subprocess, sys, json, torch
from pathlib import Path

assert globals().get('SCAN_GATE_PASS') is True, 'Cell 0 gate not passed.'
assert globals().get('TEST_GATE_PASS') is True, 'Cell 1 gate not passed.'
assert globals().get('SAFE_ACTION') in {'RESUME','START NEW CAMPAIGN'}
assert torch.cuda.is_available(), 'T4 is not available.'

# Recheck the external immutable substrate immediately before launching the child process.
missing = [name for name in ARK018_REQUIRED if not (ARK018_PREPARED / name).is_file()]
if missing:
    raise SystemExit('SAFE ACTION: STOP — ARK-018 substrate disappeared before launch: ' + ', '.join(missing))

runner = os.path.join(REPO, 'experiments/ARK-020-V4/run_ark020_v4_hardened.py')
root = Path('/content/drive/MyDrive/genisis-arkenstone/ARK020_V4_CONTINUAL')
env = dict(os.environ)
env.pop('CUDA_VISIBLE_DEVICES', None)
env['PYTHONUNBUFFERED'] = '1'
print('=== ARK-020 V4 A1.3 ===')
print('ACTION:', SAFE_ACTION)
if SAFE_ACTION == 'RESUME':
    print('Using existing Drive campaign. Completed RESULT.json arms will be skipped and RESUME.pt will be loaded automatically.')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
proc = subprocess.run([sys.executable, runner, '--mode','all'], cwd=REPO, env=env)
print('CAMPAIGN RETURN CODE:', proc.returncode)
if proc.returncode != 0:
    failure = root / 'ARK-020_V4_FAILURE.json'
    if failure.exists():
        print('\n===== REAL V4 FAILURE RECEIPT =====')
        try:
            f = json.loads(failure.read_text())
            print('EXCEPTION:', f.get('exception'))
            print('MESSAGE:', f.get('message'))
            print(f.get('traceback', failure.read_text()))
        except Exception:
            print(failure.read_text())
    raise SystemExit('ARK-020 V4 child process failed — failure receipt printed above')
result = root / 'ARK-020_V4_RESULT.json'
session = root / 'SESSION_STATE.json'
if result.exists():
    r = json.loads(result.read_text())
    print('SCIENTIFIC CAMPAIGN STATUS:', r.get('status'))
    print('VERDICT:', r.get('decision', {}).get('verdict', r.get('verdict')))
elif session.exists():
    s = json.loads(session.read_text())
    print('SESSION STATUS:', s.get('status'))
    print('MESSAGE:', s.get('message'))
    if s.get('status') == 'PARTIAL_SESSION':
        print('Expected multi-session stop. On the next T4, rerun Cell 0 -> Cell 1 -> Cell 2; it will resume again.')


In [ ]:
from pathlib import Path
import json

root = Path('/content/drive/MyDrive/genisis-arkenstone/ARK020_V4_CONTINUAL')
print('CAMPAIGN ROOT:', root)
for name in [
    'EXECUTABLE_IDENTITY_A1.json',
    'PREEXECUTION_GATE.json',
    'EXACT_RESUME_SMOKE_V4.json',
    'EXACT_RESUME_SMOKE_V4_A1.json',
    'SESSION_STATE.json',
    'ARK-020_V4_RESULT.json',
    'ARK-020_V4_FAILURE.json',
]:
    p = root / name
    if p.exists():
        print('\n===', name, '===')
        print(p.read_text()[:8000])
